## Setup 

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
import itertools

In [2]:
training_data_path = "training_data.csv"
test_data_path = "test_data.csv"

training_df = pd.read_csv(training_data_path)
X = training_df.iloc[:, :-1]
y = training_df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train.iloc[:, :] = X_train_scaled
X_test.iloc[:, :] = X_test_scaled

## SVC For Feature Selection

In [13]:
svc = LinearSVC(dual=False)
sfs = SequentialFeatureSelector(svc, n_features_to_select=4, direction='forward', n_jobs=-1)
sfs.fit(X_train, y_train)

best_features = X_train.columns[sfs.get_support()]
print(f"Selected features: {best_features}")

Selected features: Index(['distance_from_home', 'ratio_to_median_purchase_price',
       'used_pin_number', 'online_order'],
      dtype='object')


## KNN For Predictions

In [14]:
X_train_final = X_train[best_features]
X_test_final = X_test[best_features]
knn = KNeighborsClassifier(n_neighbors=3, algorithm='auto', n_jobs=-1) # This jawn needs all of my cores

knn.fit(X_train_final, y_train)
y_pred = knn.predict(X_test_final) # This takes literally so long

score = accuracy_score(y_test, y_pred)
print(f"Accuracy: {score:.4f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.9802
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99    164185
         1.0       0.90      0.87      0.89     15815

    accuracy                           0.98    180000
   macro avg       0.95      0.93      0.94    180000
weighted avg       0.98      0.98      0.98    180000



## Cross Validation

In [24]:
cv_df = pd.read_csv(test_data_path)
y_cv  = cv_df.iloc[:, -1]
X_cv = cv_df.iloc[:, :-1].copy()
X_cv_scaled = scaler.transform(X_cv)
X_cv.iloc[:, :] = X_cv_scaled

cv_pred = knn.predict(X_cv[best_features])

cv_score = accuracy_score(y_cv, cv_pred)
print(f"Accuracy: {cv_score:.4f}")
print(classification_report(y_cv, cv_pred))

Accuracy: 0.9806
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     91297
         1.0       0.90      0.87      0.89      8703

    accuracy                           0.98    100000
   macro avg       0.94      0.93      0.94    100000
weighted avg       0.98      0.98      0.98    100000



Seems like it generalized pretty well.

In [ ]:
cm = confusion_matrix(y_cv, cv_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=knn.classes_)
disp.plot(cmap="Blues", ax=ax, values_format="d")

plt.title(f'Confusion Matrix: KNN with {list(best_features)}')
plt.grid(False) # Turn off the grid for a cleaner look
plt.show()